# Task 4: Forecasting Model Development

## Objective

The objective of this task is to develop forecasting models for Ethiopia's financial inclusion indicators using the enriched dataset prepared in previous tasks. This notebook prepares the forecasting dataset, trains baseline and statistical forecasting models, evaluates model performance, and generates future forecasts that will be used in the dashboard developed in Task 5.


In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

from statsmodels.tsa.stattools import adfuller

plt.style.use("ggplot")

In [2]:
project_root = Path.cwd()

processed_path = project_root / "data" / "processed"

figures_path = project_root / "figures"

processed_path.mkdir(
    parents=True,
    exist_ok=True
)

figures_path.mkdir(
    parents=True,
    exist_ok=True
)

print(processed_path)

c:\Users\gebre\OneDrive\Documents\Programming\KIAM 10 academy\Week 11\ethiopia-fi-forecast\notebooks\data\processed


In [3]:
from pathlib import Path

print(Path.cwd())

for file in Path.cwd().rglob("*.csv"):
    print(file)

c:\Users\gebre\OneDrive\Documents\Programming\KIAM 10 academy\Week 11\ethiopia-fi-forecast\notebooks


In [4]:
from pathlib import Path

print("Current folder:")
print(Path.cwd())

print("\nSearching for CSV files...\n")

for f in Path.cwd().parent.rglob("*.csv"):
    print(f)

Current folder:
c:\Users\gebre\OneDrive\Documents\Programming\KIAM 10 academy\Week 11\ethiopia-fi-forecast\notebooks

Searching for CSV files...

c:\Users\gebre\OneDrive\Documents\Programming\KIAM 10 academy\Week 11\ethiopia-fi-forecast\models\event_indicator_association_matrix.csv
c:\Users\gebre\OneDrive\Documents\Programming\KIAM 10 academy\Week 11\ethiopia-fi-forecast\models\event_indicator_association_summary.csv
c:\Users\gebre\OneDrive\Documents\Programming\KIAM 10 academy\Week 11\ethiopia-fi-forecast\venv\Lib\site-packages\tornado\test\csv_translations\fr_FR.csv
c:\Users\gebre\OneDrive\Documents\Programming\KIAM 10 academy\Week 11\ethiopia-fi-forecast\venv\Lib\site-packages\statsmodels\tsa\vector_ar\tests\Matlab_results\test_coint.csv
c:\Users\gebre\OneDrive\Documents\Programming\KIAM 10 academy\Week 11\ethiopia-fi-forecast\venv\Lib\site-packages\statsmodels\tsa\tests\results\arima111_forecasts.csv
c:\Users\gebre\OneDrive\Documents\Programming\KIAM 10 academy\Week 11\ethiopia-fi-

In [5]:
import pandas as pd
from pathlib import Path

forecast_file = Path("../data/processed/ethiopia_fi_enriched.csv")

forecast_df = pd.read_csv(forecast_file)

forecast_df.head()

,category,collected_by,collection_date,comparable_country,confidence,evidence_basis,fiscal_year,gender,impact_direction,impact_estimate,...,region,related_indicator,relationship_type,source_name,source_type,source_url,unit,value_numeric,value_text,value_type
0,NaN,Bilen M. Gebremariam,2025-01-20,NaN,high,NaN,2014,all,NaN,NaN,...,NaN,NaN,NaN,Global Findex 2014,survey,https://www.worldbank.org/en/publication/globa...,%,22.0,NaN,percentage
1,NaN,Bilen M. Gebremariam,2025-01-20,NaN,high,NaN,2017,all,NaN,NaN,...,NaN,NaN,NaN,Global Findex 2017,survey,https://www.worldbank.org/en/publication/globa...,%,35.0,NaN,percentage
2,NaN,Bilen M. Gebremariam,2025-01-20,NaN,high,NaN,2021,all,NaN,NaN,...,NaN,NaN,NaN,Global Findex 2021,survey,https://www.worldbank.org/en/publication/globa...,%,46.0,NaN,percentage
3,NaN,Bilen M. Gebremariam,2025-01-20,NaN,high,NaN,2021,male,NaN,NaN,...,NaN,NaN,NaN,Global Findex 2021,survey,https://www.worldbank.org/en/publication/globa...,%,56.0,NaN,percentage
4,NaN,Bilen M. Gebremariam,2025-01-20,NaN,high,NaN,2021,female,NaN,NaN,...,NaN,NaN,NaN,Global Findex 2021,survey,https://www.worldbank.org/en/publication/globa...,%,36.0,NaN,percentage


In [6]:
print("Shape:", forecast_df.shape)

forecast_df.info()

forecast_df.describe(include="all")

Shape: (58, 35)
<class 'pandas.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 35 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   category             10 non-null     str    
 1   collected_by         58 non-null     str    
 2   collection_date      58 non-null     str    
 3   comparable_country   7 non-null      str    
 4   confidence           58 non-null     str    
 5   evidence_basis       14 non-null     str    
 6   fiscal_year          44 non-null     str    
 7   gender               58 non-null     str    
 8   impact_direction     14 non-null     str    
 9   impact_estimate      12 non-null     float64
 10  impact_magnitude     14 non-null     str    
 11  indicator            58 non-null     str    
 12  indicator_code       44 non-null     str    
 13  indicator_direction  34 non-null     str    
 14  lag_months           14 non-null     float64
 15  location             58 non-null     

,category,collected_by,collection_date,comparable_country,confidence,evidence_basis,fiscal_year,gender,impact_direction,impact_estimate,...,region,related_indicator,relationship_type,source_name,source_type,source_url,unit,value_numeric,value_text,value_type
count,10,58,58,7,58,14,44,58,14,12.000000,...,0.0,14,14,44,44,32,48,4.600000e+01,10,58
unique,7,1,2,4,2,3,14,3,2,NaN,...,NaN,9,3,26,7,9,9,NaN,3,6
top,product_launch,Bilen M. Gebremariam,2025-01-20,India,high,literature,2024,all,increase,NaN,...,NaN,USG_P2P_COUNT,direct,Global Findex 2021,operator,https://www.worldbank.org/en/publication/globa...,%,NaN,Launched,percentage
freq,2,58,57,3,45,7,12,54,12,NaN,...,NaN,3,9,5,15,10,27,NaN,7,29
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.416667,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.770207e+10,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.048569,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.593717e+11,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-20.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-2.000000e+01,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.750000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.425000e+01,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.500000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.550000e+01,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.775000e+06,NaN,NaN


In [7]:
missing = (
    forecast_df
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

missing[missing > 0]

region                 58
comparable_country     51
category               48
value_text             48
period_start           47
period_end             47
impact_estimate        46
impact_magnitude       44
relationship_type      44
impact_direction       44
evidence_basis         44
lag_months             44
related_indicator      44
parent_id              44
notes                  43
source_url             26
original_text          24
indicator_direction    24
source_type            14
fiscal_year            14
source_name            14
indicator_code         14
value_numeric          12
unit                   10
pillar                 10
dtype: int64

In [9]:
forecast_df["analysis_year"] = (
    pd.to_numeric(
        forecast_df["analysis_year"],
        errors="coerce"
    )
)

forecast_df = forecast_df.sort_values(
    "analysis_year"
)

forecast_df.head()

KeyError: 'analysis_year'

In [ ]:
account_df = forecast_df[
    forecast_df["indicator_code"] == "ACC_OWNERSHIP"
].copy()

account_df = account_df.sort_values(
    "analysis_year"
)

account_df[
    [
        "analysis_year",
        "value_numeric"
    ]
]

NameError: name 'forecast_df' is not defined

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    account_df["analysis_year"],
    account_df["value_numeric"],
    marker="o",
    linewidth=2
)

plt.title(
    "Historical Account Ownership"
)

plt.xlabel("Year")

plt.ylabel("Percent")

plt.grid(True)

plt.tight_layout()

plt.savefig(
    figures_path /
    "historical_account_ownership.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

NameError: name 'account_df' is not defined

<Figure size 1000x500 with 0 Axes>

In [ ]:
train_size = int(
    len(account_df) * 0.8
)

train = account_df.iloc[:train_size]

test = account_df.iloc[train_size:]

print("Training observations:", len(train))

print("Testing observations:", len(test))

NameError: name 'account_df' is not defined

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    train["analysis_year"],
    train["value_numeric"],
    marker="o",
    label="Training"
)

plt.plot(
    test["analysis_year"],
    test["value_numeric"],
    marker="o",
    label="Testing"
)

plt.title(
    "Training and Testing Split"
)

plt.xlabel("Year")

plt.ylabel("Account Ownership")

plt.legend()

plt.tight_layout()

plt.savefig(
    figures_path /
    "train_test_split.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

NameError: name 'train' is not defined

<Figure size 1000x500 with 0 Axes>

In [ ]:
adf_result = adfuller(
    account_df["value_numeric"]
)

print("ADF Statistic:", adf_result[0])

print("p-value:", adf_result[1])

if adf_result[1] < 0.05:
    print("Series is stationary.")
else:
    print("Series is not stationary.")

NameError: name 'adfuller' is not defined

In [ ]:
print("Forecast dataset loaded:", not forecast_df.empty)

print("Training rows:", len(train))

print("Testing rows:", len(test))

print(
    "Historical years:",
    account_df["analysis_year"].min(),
    "-",
    account_df["analysis_year"].max()
)

NameError: name 'forecast_df' is not defined

## Forecasting Model Development

This section develops forecasting models for the Account Ownership indicator. A baseline (Naïve) model is created and compared with an ARIMA forecasting model. Model performance is evaluated using MAE and RMSE to identify the most suitable forecasting approach.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

ModuleNotFoundError: No module named 'statsmodels'